In [2]:
import pydantic
print(pydantic.__version__)

2.13.4


In [8]:
from pydantic import BaseModel

class Person(BaseModel):
    first_name: str
    last_name: str
    age: int

In [9]:
p:Person = Person(first_name="John", last_name="Smith", age=42)
print(p.first_name)
print(p.last_name)
print(p.age)

John
Smith
42


In [10]:
print(repr(p))

Person(first_name='John', last_name='Smith', age=42)


In [ ]:
###自動型態轉換 (Type Coercion)

In [11]:
p1:Person = Person(first_name="John", last_name="Smith", age="42")
print(repr(p1))

Person(first_name='John', last_name='Smith', age=42)


In [13]:
from pydantic import ValidationError
try:
    p2:Person = Person(first_name="John", last_name="Smith", age="42a")
except ValidationError as e:
    print(e)
except Exception as e:
    print(f"不知明的錯誤{e}")

1 validation error for Person
age
  Input should be a valid integer, unable to parse string as an integer [type=int_parsing, input_value='42a', input_type=str]
    For further information visit https://errors.pydantic.dev/2.13/v/int_parsing


In [ ]:
### 反序列化(Deserializing Data)

In [14]:
data = {
    "first_name": "John",
    "last_name": "Smith",
    "age": "42"
}

p3: Person = Person.model_validate(data)
print(repr(p3))

Person(first_name='John', last_name='Smith', age=42)


In [15]:
data_json = '''
{
    "first_name": "John",
    "last_name": "Smith",
    "age":42
}
'''

p4:Person = Person.model_validate_json(data_json)
print(repr(p4))

Person(first_name='John', last_name='Smith', age=42)


In [16]:
class Person1(BaseModel):
    first_name: str
    last_name: str
    age: int = 0

In [17]:
print(Person.model_fields)

{'first_name': FieldInfo(annotation=str, required=True), 'last_name': FieldInfo(annotation=str, required=True), 'age': FieldInfo(annotation=int, required=True)}


In [18]:
p4 = Person1(first_name="john", last_name='Smith', age=10)
print(repr(p4))
p5 = Person1(first_name="john", last_name='Smith')
print(repr(p5))

Person1(first_name='john', last_name='Smith', age=10)
Person1(first_name='john', last_name='Smith', age=0)


In [19]:
class Person2(BaseModel):
    first_name: str | None = None
    last_name: str
    age: int = 0

p6 = Person2(last_name="Simit")
print(repr(p6))

Person2(first_name=None, last_name='Simit', age=0)


In [20]:
class Person3(BaseModel):
    first_name:str | None = None
    last_name: str
    age: int = 0
    lucky_numbers:list[int] = []

p7 = Person3(last_name="Smith", lucky_numbers=[1, "2", 3.0])
print(repr(p7))

Person3(first_name=None, last_name='Smith', age=0, lucky_numbers=[1, 2, 3])


In [21]:
from pydantic import Field
data = {
    "id":100,
    "First Name": "John",
    "LASTNAME": "Smith",
    "age in years": 42
}

class Person4(BaseModel):
    id_:int = Field(alias="id")
    first_name: str = Field(alias="First Name")
    last_name: str = Field(alias="LASTNAME")
    age:int = Field(alias="age in years",default=0)

p8 = Person4.model_validate(data)
print(repr(p8))

Person4(id_=100, first_name='John', last_name='Smith', age=42)


In [22]:
p8.model_dump_json()

'{"id_":100,"first_name":"John","last_name":"Smith","age":42}'

In [23]:
from datetime import datetime, timezone
from zoneinfo import ZoneInfo

datetime.now(ZoneInfo("Asia/Taipei"))

datetime.datetime(2026, 7, 17, 16, 8, 18, 85608, tzinfo=zoneinfo.ZoneInfo(key='Asia/Taipei'))

In [24]:
class Log(BaseModel):
    dt: datetime = datetime.now(ZoneInfo("Asia/Taipei"))
    message: str
log1 = Log(message="log1")
print(repr(log1))

Log(dt=datetime.datetime(2026, 7, 17, 16, 8, 21, 113158, tzinfo=zoneinfo.ZoneInfo(key='Asia/Taipei')), message='log1')


In [25]:
log2 = Log(message="log2")
print(repr(log2))

Log(dt=datetime.datetime(2026, 7, 17, 16, 8, 21, 113158, tzinfo=zoneinfo.ZoneInfo(key='Asia/Taipei')), message='log2')


In [26]:
class Log1(BaseModel):
    dt: datetime = Field(default_factory=lambda:datetime.now(ZoneInfo("Asia/Taipei")))
    message: str
log1 = Log1(message="log1")
print(repr(log1))

Log1(dt=datetime.datetime(2026, 7, 17, 16, 8, 25, 307017, tzinfo=zoneinfo.ZoneInfo(key='Asia/Taipei')), message='log1')


In [27]:
log2 = Log1(message="log2")
print(repr(log2))

Log1(dt=datetime.datetime(2026, 7, 17, 16, 8, 28, 312279, tzinfo=zoneinfo.ZoneInfo(key='Asia/Taipei')), message='log2')


In [28]:
def my_time():
    return datetime.now(ZoneInfo("Asia/Taipei"))

class Log2(BaseModel):
    dt: datetime = Field(default_factory=my_time)
    message: str
log1 = Log2(message="log1")
print(repr(log1))

Log2(dt=datetime.datetime(2026, 7, 17, 16, 8, 30, 678784, tzinfo=zoneinfo.ZoneInfo(key='Asia/Taipei')), message='log1')


In [29]:
data = {
    "firstName": "Arthur",
    "lastName": "Clarke",
    "born":{
        "place":{
            "country":"Lunar Colony",
            "city": "Central City"
        },
        "date":"2001-01-01"
    }
}

In [30]:
from datetime import date
class Place(BaseModel):
    country: str
    city: str

class Born(BaseModel):
    place:Place
    dt:date = Field(alias="date")
    
class Person(BaseModel):
    first_name:str = Field(alias="firstName")
    last_name:str = Field(alias="lastName")
    born:Born

p10 = Person.model_validate(data)
print(repr(p10))

Person(first_name='Arthur', last_name='Clarke', born=Born(place=Place(country='Lunar Colony', city='Central City'), dt=datetime.date(2001, 1, 1)))


In [31]:
p10.model_dump_json()

'{"first_name":"Arthur","last_name":"Clarke","born":{"place":{"country":"Lunar Colony","city":"Central City"},"dt":"2001-01-01"}}'

In [ ]:
data = [
    {
    "year": "114",
    "bureau": "蘆洲分局",
    "unit": "更寮派出所",
    "location": "四維路5號",
    "build_case_unit": "自建案"
    },
    {
    "year": "114",
    "bureau": "蘆洲分局",
    "unit": "更寮派出所",
    "location": "四維路5號",
    "build_case_unit": "自建案"
    }
]

In [ ]:
from pydantic import TypeAdapter
import requests

class VideoPosition(BaseModel):
    year:str
    bureau:str
    unit:str
    location:str
    build_case_unit: str

url = 'https://data.ntpc.gov.tw/api/datasets/1b72abe8-8862-4130-aeb8-178c1240e6c4/json?page=0&size=1000'
response = requests.get(url)
data = response.json()

adapter = TypeAdapter(list[VideoPosition])
list_position = adapter.validate_python(data)

list_position